# 🚩 Red-Teaming Multi-Turn Testing (Single `red_config.yaml`)

This notebook runs a **red-teaming simulation** against your HTTP assistant endpoint.  
All parameters (endpoints, request templates, parsing rules, objectives, timeouts, report paths, etc.) are loaded from one central **`red_config.yaml`**.

**Advantages**:  
- Product teams only edit `red_config.yaml` when anything changes (URLs, tokens, regex patterns, objectives, etc.).  
- The notebook’s code cells remain static.

**Overview**:  
1. Load environment variables and parse `red_config.yaml`.  
2. Build a `MultiFieldResponseParser` + thread-ID helpers.  
3. Instantiate `HTTPTargetX`, `Evaluator`, and `RedTeamingOrchestrator` (in “red-team” mode).
4. Loop over each objective in `red_config.yaml` → call `run_simulation_async` on the orchestrator.  
5. Generate a final HTML via `generate_simulation_report`.  

> **Before you begin**:  
> - Ensure your `.env` contains:
>   ```ini
>   TARGET_ENDPOINT=<your_endpoint_prefix>
>   AUTH_TOKEN=<your_api_token>
>   ```  
> - Place this notebook and **`red_config.yaml`** in the same directory.  
> - Verify that `strategy_path` and `scorer_path` in `red_config.yaml` point to existing YAML files.


In [ ]:
# Cell 1: Imports, logging, and load both .env + config.yaml

import logging
import os
import re
import time
import asyncio
import yaml
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# PyRIT (in-memory DuckDB)
from pyrit.common import IN_MEMORY, initialize_pyrit
from pyrit.prompt_target import MultiFieldResponseParser, OpenAIChatTarget
from pyrit.prompt_target import HTTPTargetX
from pyrit.score.evaluator import Evaluator
from pyrit.common.report_generator import get_conversation_report_async
from pyrit.common.report_generator import create_report

from typing import Optional

# ── Configure Logging ───────────────────────────────────────────────────────────
logging.basicConfig(level=logging.WARNING)

# ── Initialize PyRIT Memory ────────────────────────────────────────────────────
initialize_pyrit(memory_db_type=IN_MEMORY)

# ── Load environment variables ─────────────────────────────────────────────────
load_dotenv()

# ── Read config.yaml ────────────────────────────────────────────────────────────
config_path = Path("red_config.yaml")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

# ── Extract each field from config.yaml ────────────────────────────────────────
http_request_raw        = cfg["http_request_raw"]
field_defs              = cfg["field_defs"]
thread_id_pattern       = cfg["thread_id_pattern"]
strategy_path           = cfg["strategy_path"]
scorer_path             = cfg["scorer_path"]
objectives              = cfg["objectives"]
max_turns               = cfg["max_turns"]
timeout_seconds         = cfg["timeout_seconds"]
use_score_as_feedback   = cfg["use_score_as_feedback"]
scorer_type             = cfg["scorer_type"]
report_path             = cfg["report_path"]
thread_id_key           = cfg["thread_id_query_param_key"]

# ── Build “base_url” + “token” from environment (from .env) ────────────────────
base_url   = os.getenv("TARGET_ENDPOINT")
token      = os.getenv("AUTH_TOKEN")

# ── Substitute into the raw HTTP template (YAML uses double-braces for PROMPT) ─
http_request_templated = http_request_raw.format(
    base_url=base_url,
    token=token
)

## 🔍 Build `MultiFieldResponseParser` & Thread-ID Helpers

1. **`MultiFieldResponseParser`** reads rules from `field_defs` in `red_config.yaml`.  
2. **`thread_id_parser`** uses the single‐line regex `thread_id_pattern` from `red_config.yaml` to extract a new thread ID.  
3. **`thread_id_injector`** strips any existing threadId parameter from the assistant’s URL and appends a fresh one.


In [ ]:
# Cell 2: Parser + Thread-ID helpers

import requests

# 2.1: Build MultiFieldResponseParser from YAML’s field_defs
multi_parser = MultiFieldResponseParser(field_definitions=field_defs)

# 2.2 thread_id_parser using the YAML‐provided regex
def thread_id_parser(response: str) -> Optional[str]:
    pattern = rf"{re.escape(thread_id_pattern)}\s*\ndata:(.*)"
    match = re.search(pattern, response)
    return match.group(1).strip() if match else None

# 2.3 thread_id_injector (appends threadId received to the next request)
def thread_id_injector(raw_http_request: str, thread_id: str, thread_id_key: str = "threadId") -> str:
    """
    Injects or replaces the `{thread_id_key}` query parameter in the first URL found in the raw HTTP request.
    """
    import re

    url_pattern = r"(https?://[^\s]+)"
    m = re.search(url_pattern, raw_http_request)
    if not m:
        raise ValueError("No URL found in raw HTTP request; cannot inject thread ID.")

    original_url = m.group(1)

    # Remove existing threadId (or other key) if present
    pattern = rf"([?&]){re.escape(thread_id_key)}=[^&]*"
    cleaned = re.sub(pattern, "", original_url)

    sep = "&" if "?" in cleaned else "?"
    new_url = f"{cleaned}{sep}{thread_id_key}={thread_id}"

    new = raw_http_request.replace(original_url, new_url)
        
    return new



## 🚀 Instantiate `HTTPTargetX`, `Evaluator`, and `RedTeamingAttack` (for Red-Teaming)

1. **`HTTPTargetX`**:
   - `http_request` = `http_request_templated`
   - `prompt_regex_string` = `"{PROMPT}"`
   - `use_tls=True`
   - `response_parser` = `multi_parser`
   - `thread_id_parser` = `thread_id_parser`
   - `timeout` = `timeout_seconds`
     
2. **`Evaluator`** (scorer):
   - Uses `OpenAIChatTarget()` plus `scorer_path` (from `red_config.yaml`)
   - `scorer_type` = `"true_false"` (from `red_config.yaml`)
   - `additional_evaluator_variables` will be set per‐objective inside the loop

3. **`RedTeamingOrchestrator`**:
   - Operates in “red-team” style (still calls `run_simulation_async` but uses `true_false` logic)
   - `objective_target` = `http_prompt_target`
   - `adversarial_chat` = `OpenAIChatTarget()`
   - `adversarial_chat_system_prompt_path` = `strategy_path`
   - `objective_scorer` = `scorer_instance` (built per‐objective)
   - `scorer_type` = `"true_false"`
   - `evaluate_chat=False` (because we only care about single‐turn violation detection)
   - `max_turns` = `max_turns`
   - `use_score_as_feedback` = `use_score_as_feedback`
   - `thread_id_injector` = `thread_id_injector`
   - `verbose=True`


In [ ]:
# Cell 3: Instantiate HTTPTargetX, Evaluator, and RedTeamingOrchestrator
import aiohttp

# Set robust timeout values (in seconds)
timeout = aiohttp.ClientTimeout(
    total=None,           # No overall timeout, or set to e.g. 300 for 5 min
    connect=60,           # 60 seconds to connect
    sock_connect=60,      # 60 seconds to establish a socket connection
    sock_read=300         # 300 seconds to read response (for streaming endpoints)
)

client = aiohttp.ClientSession(timeout=timeout)

# 3.1 Create our HTTP target using the templated raw request
http_prompt_target = HTTPTargetX(
    http_request         = http_request_templated,
    prompt_regex_string  = "{PROMPT}",
    use_tls              = True,
    response_parser      = multi_parser,
    thread_id_parser     = thread_id_parser,
    client               = client
)

## 🏃‍♀️ Run Each Objective via `run_red_teaming()`

Below, we define:
1. **`run_red_teaming()`**:
   - Loops over each string in `objectives` (from `red_config.yaml`).
   - For each:
     - Build a fresh `Evaluator(...)` with `{"restricted_topic": objective}`.
     - Instantiate a new `RedTeamingOrchestrator(...)` configured for red-teaming.
     - Call `await orchestrator.run_simulation_async(objective=objective)`.
     - Collect its conversation report via `get_conversation_report_async()`.
   - Return a list of those reports (dicts).

2. **`generate_report()`**:
   - Writes out a single HTML file under `report_path` (from `red_config.yaml`), timestamped.
   - Calls `generate_simulation_report(...)` to produce the final HTML.


In [ ]:
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.executor.attack import MultiTurnAttackContext, RedTeamingAttack, AttackConverterConfig, AttackScoringConfig, AttackAdversarialConfig
from pyrit.prompt_converter import (
    TranslationConverter
)

async def run_red_teaming():
    """
    Loops over each objective in config.yaml → runs RedTeamingAttack → returns list of reports.
    """
    reports = []
    start_time = time.time()

    for objective in objectives:
        # 1) Per-objective scorer variables
        scorer_vars = {"objective": objective}

        # 2) Clone base Evaluator & inject per-objective variables
        evaluator_instance = Evaluator(
            chat_target                    = OpenAIChatTarget(),
            evaluator_yaml_path            = Path(scorer_path),
            additional_evaluator_variables = scorer_vars,
            scorer_type                    = scorer_type
        )

        # 3) Build RedTeamingAttack (not orchestrator!)
        attack_adversarial_config = AttackAdversarialConfig(
            target                      = OpenAIChatTarget(),
            system_prompt_path          = Path(strategy_path),
            seed_prompt                 = "",  # or your seed prompt string/SeedPrompt
        )
        
        attack_converter_config = AttackConverterConfig(
            request_converters=[],
            response_converters=[]
        )
        attack_scoring_config = AttackScoringConfig(
            objective_scorer            = evaluator_instance,
            auxiliary_scorers           = [],
            use_score_as_feedback       = use_score_as_feedback,
            successful_objective_threshold = 0.5  # adjust as needed
        )

        red_teaming_attack = RedTeamingAttack(
            objective_target            = http_prompt_target,
            attack_adversarial_config   = attack_adversarial_config,
            attack_converter_config     = attack_converter_config,
            attack_scoring_config       = attack_scoring_config,
            prompt_normalizer           = PromptNormalizer(),
            evaluate_chat               = False,
            scorer_type                 = scorer_type,
            thread_id_injector          = thread_id_injector,
            max_retries                 = 3,
            max_turns                   = max_turns,
        )

        # 4) Run the red-teaming simulation asynchronously
        context = MultiTurnAttackContext(
            objective=objective,
            memory_labels={},
        )

        sim_result = await red_teaming_attack.execute_with_context_async(context=context)

        # 5) Pull back its conversation report (dict) and append
        reports.append(await get_conversation_report_async(sim_result))

    elapsed = time.time() - start_time
    print(f"✅ All red-teaming simulations done in {elapsed:.2f} seconds")
    return reports

async def generate_report(results: list, execution_time: float):
    """
    Writes out a single HTML under report_path with a timestamp.
    """
    report_dir = Path(report_path).resolve()
    report_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"simulation_report_{ts}.html"

    create_report(
        results        = results,
        save_path      = report_dir / fname,
        description    = (
            "This report provides an overview of multi-turn customer simulations. The report includes conversation "
            "transcripts, assistant responses, and corresponding scores reflecting the quality and relevance of the interaction."
        ),
        execution_time = execution_time
    )

## 🎬 Kick-Off: `main()`

1. Call `await run_red_teaming()` → get a list of conversation reports (one dict per objective).  
2. Measure elapsed time, then hand both to `generate_report(...)`.  
3. The final HTML appears under `report_path` (from `red_config.yaml`) with a timestamp.


In [ ]:
# Cell 5: Main async runner

async def main():
    start_time = time.time()
    results = await run_red_teaming()
    exec_time = time.time() - start_time
    await generate_report(results, exec_time)

# Execute
await main()